# SIS #3 — The Warehouse Sequence Detective
## RNN vs. LSTM: Does the order of events matter?

**CSCI1409 · Deep Learning · Bachelor’s degree · Week 11 · 4 points**
**Format:** In-class team assignment, followed by an individual oral defense
**Team size:** 1–3 students · **Submission:** one completed `.ipynb` per team on Microsoft Teams

### Scenario
A warehouse camera system detects these five actions:

| Event ID | Action |
|:--:|---|
| `0` | IDLE |
| `1` | CHECK |
| `2` | MOVE |
| `3` | SCAN |
| `4` | PACK |

Your task is to classify a sequence of events:

- **Label `1` (correct):** `SCAN` occurred **before** `PACK`.
- **Label `0` (incorrect):** `PACK` occurred **before** `SCAN`.

Every sequence contains **exactly one SCAN and one PACK**, plus unrelated actions. So knowing *which* actions occurred is not enough: your model must learn **their order**.

**Example:** `IDLE → SCAN → MOVE → PACK` is class **1**, whereas `IDLE → PACK → MOVE → SCAN` is class **0**.

### What is provided
The **first code cell below is the entire provided dataset** and the only provided code. Run it once without changing it. It creates the train, validation and test sets in memory, using only NumPy. **No downloads, external files, data cleaning, or GPU are needed.**

The input arrays are `X_train`, `X_val`, `X_test` and the labels are `y_train`, `y_val`, `y_test`. Each input is a sequence of **16 one-hot event vectors** (5 features per time step).

### Your task — write ALL code yourselves
You must implement **every model, training loop, metric, and experiment from scratch**, using the **blank code cells** provided after the dataset. You may use standard PyTorch building blocks such as `nn.Linear`, `nn.RNN`, `nn.LSTM`, `CrossEntropyLoss`, `Adam` and `DataLoader`. You do **not** need to implement LSTM gates manually.

- **A. Explore + pipeline:** Inspect the data, then write reusable training and evaluation functions.
- **B. Baseline:** Train an order-blind model.
- **C. RNN:** Build and train a vanilla RNN classifier.
- **D. LSTM:** Build and train an LSTM classifier.
- **E. Test + reversal:** Evaluate all models on the test set once, then reverse the test sequences and compare predictions.
- **F. Conclusions:** Report your results and answer the short questions.
- **G. Optional challenges** (not graded).

Every model trains in a few seconds on a laptop CPU. Spend your time on **predicting** results before you run, **reading** your metrics, and **explaining** why each model behaves as it does: that is what the questions and the defense are about.

### Reproducibility
- Define `SEED = 0`. Call `torch.manual_seed(SEED)` **immediately before creating each model**, and seed the mini-batch shuffling, e.g. `DataLoader(..., shuffle=True, generator=torch.Generator().manual_seed(SEED))`.
- The numbers in your Part F table must be the ones your notebook prints after **Kernel → Restart & Run All**.

### Rules and AI policy
This assignment follows the course’s **Level C (AI-Assisted)** policy.

- **Chat-based AI is allowed** for explanations, debugging and individual code snippets. You must **write, assemble, run, verify and understand** your own complete solution.
- **AI coding agents and code-editing assistants are not allowed:** no Codex, Claude Code, IDE-integrated coding agents, or tools that automatically create/edit/run your notebook for you. Use AI **only through a chat interface**.
- Cite any code you adapt from outside sources in a short comment. Verify everything an AI tells you: you will be asked to explain it.
- Do not change the provided dataset, labels or splits. Train on **train only**. In Parts A–D, check your models with the **validation** set only. The **test** set is used **once**, in Part E, after all models are trained.
- **All team members must participate and can be questioned on any part of the code**, not just the part they personally wrote.
- You are graded on correct implementation and understanding, **not** on achieving a particular accuracy threshold. Show your results even if a model performs poorly.

### Grading — 4.0 points

| Part | What you deliver | Points |
|---|---|---:|
| A | Data exploration; reusable training and evaluation functions | **0.50** |
| B | Order-blind baseline, trained and explained (F-Q1) | **0.50** |
| C | Vanilla RNN, trained and explained (F-Q2) | **0.75** |
| D | LSTM, trained and explained (F-Q3, F-Q4) | **0.75** |
| E | Test evaluation, reversal experiment and interpretation (F-Q5) | **0.50** |
| **Oral defense** | Individual understanding of the code and concepts | **1.00** |
| **Total** | | **4.00** |

Each part is graded on correct code, a reported result, and a correct explanation. Part G is optional and not graded.

**Suggested settings** (use the **same** settings for all models): `hidden_size=24`, `batch_size=128`, `epochs=10`, `learning_rate=0.01`, one recurrent layer. Use **2 output logits** for the two classes and cross-entropy loss.

---

### PROVIDED DATASET — run the following cell without editing it


In [8]:
import numpy as np

EVENTS = ["IDLE", "CHECK", "MOVE", "SCAN", "PACK"]
SEQUENCE_LENGTH = 16

def _make_split(n_samples, seed):
    rng = np.random.default_rng(seed)
    event_ids = rng.integers(0, 3, size=(n_samples, SEQUENCE_LENGTH), dtype=np.uint8)
    labels = np.array([0, 1] * (n_samples // 2), dtype=np.int64)
    rng.shuffle(labels)
    for i in range(n_samples):
        first, second = sorted(rng.choice(SEQUENCE_LENGTH, size=2, replace=False))
        event_ids[i, first], event_ids[i, second] = (
            (3, 4) if labels[i] == 1 else (4, 3)
        )
    return np.eye(5, dtype=np.float32)[event_ids], labels

X_train, y_train = _make_split(1200, seed=4)
X_val, y_val = _make_split(240, seed=5)
X_test, y_test = _make_split(240, seed=6)

print("Dataset ready — zero downloads required")
print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)


Dataset ready — zero downloads required
Train: (1200, 16, 5) (1200,)
Val:   (240, 16, 5) (240,)
Test:  (240, 16, 5) (240,)


## A — Explore the data and build the pipeline

### A1. Explore

**Write your own Python code** to:

1. Print the input shape and class counts for the train split.
2. Display **one example with label 1** and **one example with label 0** as readable event names, using the `EVENTS` list.
3. Inspect one time-step vector (for example, the first vector in the first sequence).

**Think about:** What do `N`, `T` and `5` represent in a shape `(N, T, 5)`? Why can two sequences containing the same number of SCAN and PACK events have different labels?

In [9]:
print("Train shape:", X_train.shape)
print("Label 0 count:", (y_train == 0).sum())
print("Label 1 count:", (y_train == 1).sum())

def show(i):
    ids = X_train[i].argmax(axis=1)
    names = [EVENTS[k] for k in ids]
    print("label", y_train[i], ":", names)

show(np.where(y_train == 1)[0][0])
show(np.where(y_train == 0)[0][0])

print("First time-step vector:", X_train[0, 0])


Train shape: (1200, 16, 5)
Label 0 count: 600
Label 1 count: 600
label 1 : ['IDLE', 'IDLE', 'MOVE', 'MOVE', 'MOVE', 'IDLE', 'SCAN', 'MOVE', 'MOVE', 'IDLE', 'CHECK', 'MOVE', 'MOVE', 'PACK', 'MOVE', 'CHECK']
label 0 : ['MOVE', 'CHECK', 'MOVE', 'MOVE', 'CHECK', 'IDLE', 'MOVE', 'PACK', 'IDLE', 'MOVE', 'CHECK', 'MOVE', 'SCAN', 'MOVE', 'MOVE', 'IDLE']
First time-step vector: [1. 0. 0. 0. 0.]


**Answers A1**

- `N` = number of sequences, `T` = 16 time steps, `5` = one-hot vector for 5 event types.
- Every sequence has exactly one SCAN and one PACK. The label depends only on **which comes first**, so counts are the same for both labels.

### A2. Reusable training and evaluation functions

Using **PyTorch**, implement your own code that:

- Converts the provided NumPy arrays to suitable PyTorch tensors (which dtype does `CrossEntropyLoss` expect for the labels?).
- Trains a model on the **training split** using mini-batches, cross-entropy loss and Adam, with seeded shuffling.
- Performs the normal training steps: forward pass, loss, clear old gradients, backward pass and optimizer update.
- Computes **accuracy** without updating the model: use `model.eval()` and `torch.no_grad()`.
- Prints the training loss and the **validation** accuracy after every epoch.

*You will reuse these functions in Parts B–E.*

In [10]:
import torch
import torch.nn as nn

Xtr, ytr = torch.tensor(X_train), torch.tensor(y_train)
Xva, yva = torch.tensor(X_val),   torch.tensor(y_val)
Xte, yte = torch.tensor(X_test),  torch.tensor(y_test)
print(Xtr.dtype, ytr.dtype)

EPOCHS = 30
LR = 0.01
BATCH = 32

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = model(X).argmax(dim=1)
    return (pred == y).float().mean().item()

def train(model):
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    g = torch.Generator().manual_seed(0)
    for epoch in range(EPOCHS):
        model.train()
        order = torch.randperm(len(Xtr), generator=g)
        total = 0
        for start in range(0, len(Xtr), BATCH):
            idx = order[start:start + BATCH]
            xb, yb = Xtr[idx], ytr[idx]
            out = model(xb)
            loss = loss_fn(out, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        print(f"Epoch {epoch+1:2d} | loss {total/len(Xtr):.4f} | val acc {accuracy(model, Xva, yva):.3f}")


torch.float32 torch.int64


## B — Order-blind baseline

Create a small model that first **averages the 16 input vectors over the time dimension** and then applies a linear layer mapping **5 features → 2 class logits**.

Train it with your functions from Part A and report its **validation** accuracy. This model knows the **counts/proportions of events**, but not their order.

**Predict before you run:** can this model learn SCAN-before-PACK? Give a reason based on the *information lost by averaging*, then check your prediction.

In [11]:
class Baseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(5, 2)
    def forward(self, x):
        x = x.mean(dim=1)
        return self.fc(x)

torch.manual_seed(0)
baseline = Baseline()
train(baseline)
base_val = accuracy(baseline, Xva, yva)
print("Baseline validation accuracy:", round(base_val, 3))


Epoch  1 | loss 0.6941 | val acc 0.508
Epoch  2 | loss 0.6932 | val acc 0.517
Epoch  3 | loss 0.6934 | val acc 0.500
Epoch  4 | loss 0.6941 | val acc 0.479
Epoch  5 | loss 0.6951 | val acc 0.479
Epoch  6 | loss 0.6930 | val acc 0.496
Epoch  7 | loss 0.6926 | val acc 0.508
Epoch  8 | loss 0.6954 | val acc 0.463
Epoch  9 | loss 0.6939 | val acc 0.508
Epoch 10 | loss 0.6936 | val acc 0.454
Epoch 11 | loss 0.6928 | val acc 0.483
Epoch 12 | loss 0.6924 | val acc 0.450
Epoch 13 | loss 0.6934 | val acc 0.450
Epoch 14 | loss 0.6931 | val acc 0.496
Epoch 15 | loss 0.6935 | val acc 0.496
Epoch 16 | loss 0.6929 | val acc 0.454
Epoch 17 | loss 0.6937 | val acc 0.450
Epoch 18 | loss 0.6929 | val acc 0.454
Epoch 19 | loss 0.6928 | val acc 0.446
Epoch 20 | loss 0.6928 | val acc 0.450
Epoch 21 | loss 0.6937 | val acc 0.500
Epoch 22 | loss 0.6943 | val acc 0.488
Epoch 23 | loss 0.6938 | val acc 0.450
Epoch 24 | loss 0.6937 | val acc 0.454
Epoch 25 | loss 0.6927 | val acc 0.450
Epoch 26 | loss 0.6945 | 

**Prediction:** No. After averaging, SCAN-first and PACK-first look exactly the same (one SCAN, one PACK), so order is lost. We expect about 50% (guessing). The result above confirms this.

## C — Vanilla RNN classifier

Implement a **new** PyTorch model using:

- A **single-layer `nn.RNN`** with `input_size=5`, `hidden_size=24`, `batch_first=True`.
- The **final hidden state** as the sequence representation.
- A linear classification head from **24 hidden features → 2 class logits**.

Train it with your functions from Part A and report its **validation** accuracy.

**Shape checkpoint:** With batch size `B`, the input is `(B, 16, 5)`; for a one-layer unidirectional RNN, the final hidden-state tensor is `(1, B, 24)`; the classifier output must be `(B, 2)`. **Print these three shapes for one small batch.**

**Defense checkpoint:** What information must the final hidden state carry for this task?

In [12]:
class RNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size=5, hidden_size=24, batch_first=True)
        self.fc = nn.Linear(24, 2)
    def forward(self, x):
        outputs, h_n = self.rnn(x)
        return self.fc(h_n[0])

torch.manual_seed(0)
rnn = RNNModel()

xb = Xtr[:8]
_, h = rnn.rnn(xb)
print("input:", xb.shape)
print("h_n:", h.shape)
print("logits:", rnn(xb).shape)

train(rnn)
rnn_val = accuracy(rnn, Xva, yva)
print("RNN validation accuracy:", round(rnn_val, 3))


input: torch.Size([8, 16, 5])
h_n: torch.Size([1, 8, 24])
logits: torch.Size([8, 2])
Epoch  1 | loss 0.6024 | val acc 0.892
Epoch  2 | loss 0.2281 | val acc 0.908
Epoch  3 | loss 0.1398 | val acc 0.971
Epoch  4 | loss 0.0533 | val acc 0.992
Epoch  5 | loss 0.0129 | val acc 1.000
Epoch  6 | loss 0.0074 | val acc 1.000
Epoch  7 | loss 0.0690 | val acc 0.983
Epoch  8 | loss 0.1720 | val acc 0.996
Epoch  9 | loss 0.0400 | val acc 1.000
Epoch 10 | loss 0.0052 | val acc 1.000
Epoch 11 | loss 0.0027 | val acc 1.000
Epoch 12 | loss 0.0020 | val acc 1.000
Epoch 13 | loss 0.0015 | val acc 1.000
Epoch 14 | loss 0.0013 | val acc 1.000
Epoch 15 | loss 0.0011 | val acc 1.000
Epoch 16 | loss 0.0009 | val acc 1.000
Epoch 17 | loss 0.0008 | val acc 1.000
Epoch 18 | loss 0.0007 | val acc 1.000
Epoch 19 | loss 0.0006 | val acc 1.000
Epoch 20 | loss 0.0005 | val acc 1.000
Epoch 21 | loss 0.0005 | val acc 1.000
Epoch 22 | loss 0.0004 | val acc 1.000
Epoch 23 | loss 0.0004 | val acc 1.000
Epoch 24 | loss 0.

**Defense:** The final hidden state must remember **whether SCAN came before PACK** (what it saw first), because it only sees the sequence step by step.

## D — LSTM classifier

Implement a **new** PyTorch model using:

- A **single-layer `nn.LSTM`** with `input_size=5`, `hidden_size=24`, `batch_first=True`.
- The **final hidden state `h_n`**, not the cell state `c_n`, as the input to the classifier.
- A linear classification head from **24 hidden features → 2 class logits**.

Train it with **exactly the same settings as the RNN** and report its **validation** accuracy.

**API checkpoint:** The RNN returns `(outputs, h_n)`, while the LSTM returns `(outputs, (h_n, c_n))`.

**Defense checkpoint:** In simple words, what is the purpose of `c_n`? How is it different from `h_n`? **You do not need to implement the gates from scratch.**

In [13]:
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=5, hidden_size=24, batch_first=True)
        self.fc = nn.Linear(24, 2)
    def forward(self, x):
        outputs, (h_n, c_n) = self.lstm(x)
        return self.fc(h_n[0])

torch.manual_seed(0)
lstm = LSTMModel()
train(lstm)
lstm_val = accuracy(lstm, Xva, yva)
print("LSTM validation accuracy:", round(lstm_val, 3))


Epoch  1 | loss 0.5875 | val acc 0.892
Epoch  2 | loss 0.1824 | val acc 0.933
Epoch  3 | loss 0.0697 | val acc 0.975
Epoch  4 | loss 0.0431 | val acc 0.988
Epoch  5 | loss 0.0055 | val acc 1.000
Epoch  6 | loss 0.0017 | val acc 1.000
Epoch  7 | loss 0.0010 | val acc 1.000
Epoch  8 | loss 0.0013 | val acc 1.000
Epoch  9 | loss 0.0006 | val acc 1.000
Epoch 10 | loss 0.0005 | val acc 1.000
Epoch 11 | loss 0.0004 | val acc 1.000
Epoch 12 | loss 0.0003 | val acc 1.000
Epoch 13 | loss 0.0002 | val acc 1.000
Epoch 14 | loss 0.0002 | val acc 1.000
Epoch 15 | loss 0.0002 | val acc 1.000
Epoch 16 | loss 0.0002 | val acc 1.000
Epoch 17 | loss 0.0001 | val acc 1.000
Epoch 18 | loss 0.0001 | val acc 1.000
Epoch 19 | loss 0.0001 | val acc 1.000
Epoch 20 | loss 0.0001 | val acc 1.000
Epoch 21 | loss 0.0001 | val acc 1.000
Epoch 22 | loss 0.0001 | val acc 1.000
Epoch 23 | loss 0.0001 | val acc 1.000
Epoch 24 | loss 0.0001 | val acc 1.000
Epoch 25 | loss 0.0001 | val acc 1.000
Epoch 26 | loss 0.0001 | 

**Defense:** `c_n` is the LSTM's long-term memory: gates decide what to keep or forget in it. `h_n` is the short-term output at the last step, which we give to the classifier.

## E — Final test evaluation and the reversal experiment

All three models are now trained. Do not change any model or setting from here on.

1. Evaluate the baseline, RNN and LSTM on the **test set, once**. These are your reported test accuracies.
2. Reverse each test sequence **along the time dimension only** (keep its 5 features intact).
3. Check that the number of each type of event is unchanged by reversal.
4. Evaluate all three models on the reversed inputs **against the original `y_test` labels**. Label this as a **diagnostic**, not normal test accuracy.
5. Compare predictions: compute the **flip rate**, the fraction of test sequences whose predicted class changes after reversal.
6. Reason about the **correct** labels for the reversed sequences. (Hint: every sequence contains exactly one SCAN and one PACK.)

**Before you conclude anything:** a model can score close to 0% in step 4. Think about what that number means before you decide whether the model is broken.

**Questions:** Which models respond when the order changes? Why might the order-blind model give the same predictions after reversal?

In [14]:
models = {"Baseline": baseline, "RNN": rnn, "LSTM": lstm}

Xrev = torch.flip(Xte, dims=[1])

print("Counts unchanged:", torch.equal(Xte.sum(dim=1), Xrev.sum(dim=1)))

results = {}
for name, m in models.items():
    m.eval()
    with torch.no_grad():
        p_orig = m(Xte).argmax(dim=1)
        p_rev = m(Xrev).argmax(dim=1)
    test_acc = (p_orig == yte).float().mean().item()
    rev_acc = (p_rev == yte).float().mean().item()
    rev_correct = (p_rev == 1 - yte).float().mean().item()
    flip = (p_orig != p_rev).float().mean().item()
    results[name] = (test_acc, rev_acc, flip)
    print(f"{name:8s} | test acc {test_acc:.3f} | DIAGNOSTIC reversed vs original labels {rev_acc:.3f} "
          f"| reversed vs correct (flipped) labels {rev_correct:.3f} | flip rate {flip:.3f}")


Counts unchanged: True
Baseline | test acc 0.508 | DIAGNOSTIC reversed vs original labels 0.508 | reversed vs correct (flipped) labels 0.492 | flip rate 0.000
RNN      | test acc 1.000 | DIAGNOSTIC reversed vs original labels 0.000 | reversed vs correct (flipped) labels 1.000 | flip rate 1.000
LSTM     | test acc 1.000 | DIAGNOSTIC reversed vs original labels 0.000 | reversed vs correct (flipped) labels 1.000 | flip rate 1.000


**Answers E**

- Reversing swaps the order of SCAN and PACK, so the **correct** label of a reversed sequence is the opposite: `1 - y_test`.
- That is why a good order-aware model gets almost 0% against the original labels — it is correct, not broken.
- RNN and LSTM change their answers (high flip rate). The baseline averages over time, and the average is the same after reversal, so it gives the same predictions (flip rate 0).

## F — Results and conclusions

### Results table

| Model | Validation accuracy | Test accuracy | Reversed vs. original labels (diagnostic) | Flip rate |
|---|---:|---:|---:|---:|
| Order-blind baseline | 0.450 | 0.508 | 0.508 | 0.000 |
| Vanilla RNN | 1.000 | 1.000 | 0.000 | 1.000 |
| LSTM | 1.000 | 1.000 | 0.000 | 1.000 |

### Short answers

**1.** Why can a model that averages events over time not reliably distinguish SCAN-before-PACK from PACK-before-SCAN?

**Answer:** Averaging only keeps how many times each event appears, not when. Both classes have one SCAN and one PACK, so their averages look the same and the model can only guess (~50%).

**2.** In `(B, T, 5)`, what are `B` and `T`? What is the shape of the final output logits for `B = 8`? Why?

**Answer:** `B` is the batch size (number of sequences), `T` is the number of time steps (16). For `B = 8` the logits are `(8, 2)`: one row per sequence and one score for each of the 2 classes.

**3.** What is the difference between the hidden state `h_n` and the LSTM cell state `c_n`?

**Answer:** `c_n` is the long-term memory inside the LSTM, controlled by gates (keep / forget). `h_n` is the output of the last step, built from `c_n`, and it is what we pass to the classifier.

**4.** Which model worked best in your experiment? Is the difference between the RNN and the LSTM large enough to matter here? Does this mean LSTM always beats a vanilla RNN?

**Answer:** RNN and LSTM both got 100% on validation and test; the LSTM only learned a bit faster. The task is short (16 steps), so a simple RNN is enough. LSTM does not always win — it helps mostly with long sequences where an RNN forgets.

**5.** After reversal, an order-aware model can score close to 0% against the original labels. Is that a failure?

**Answer:** No. Reversal swaps SCAN and PACK, so the correct label becomes the opposite. RNN and LSTM got 0% vs. original labels and 100% vs. flipped labels, with flip rate 1.0 — they really use order. The baseline has flip rate 0, so it ignores order.

### Team members and contributions

- **Student 1** — Full name / student ID / contribution: …
- **Student 2** — Full name / student ID / contribution: …
- **Student 3** — Full name / student ID / contribution: …

### Before submitting

- **Kernel → Restart & Run All** finishes without errors, and the numbers in your Part F table match the printed outputs.
- The test set is evaluated only in Part E.
- Each team member can explain any chosen code line during the **individual oral defense**.

**Submission:** One completed `.ipynb` per team via Microsoft Teams. No separate report required.